## 12. Next Steps & Customization Guide

### Data Setup Instructions

1. **Create the data folder structure:**
   ```
   ./data/commodity_data/
   ├── wood.csv
   ├── paper.csv
   ├── scrap.csv
   ├── building_materials.csv
   ├── metals.csv
   ├── chemicals.csv
   ├── ... (14 total)
   ```

2. **CSV file format (required columns):**
   - `year`: 4-digit year (e.g., 2020)
   - `month`: month of year (1-12)
   - `week`: calendar week of year (1-53)
   - `ttokm`: thousand ton-kilometers (numeric)

3. **Update commodity list** if your 14 groups differ from the defaults.

### Model Configuration

- **FORECAST_WEEKS**: Currently set to 26 weeks (~6 months). Adjust for 3-18 month range:
  - 3 months ≈ 13 weeks
  - 18 months ≈ 78 weeks

- **TEST_SIZE**: Reserved data for validation (52 weeks = 1 year)

- **Prophet**: Best for trends + seasonality. Adjust `changepoint_prior_scale` (0.03-0.1) for flexibility.

- **SARIMAX**: Auto-detects seasonal patterns. May require manual tuning of (P,D,Q,s) if auto_arima struggles.

- **LSTM**: Deep learning approach. Increase `epochs` or add layers for complex patterns.

### Performance Metrics

- **RMSE**: Penalizes large errors (good for outliers)
- **MAE**: Average absolute error (interpretable unit)
- **MAPE**: Percentage error (good for comparing across scales)

### Troubleshooting

- No test data: Check `TEST_SIZE` vs. actual data length
- SARIMAX convergence issues: Reduce max_p/max_q values or enable seasonal=False
- LSTM requires minimum ~100 data points. Use Prophet/SARIMAX if data is sparse
- Memory issues with large datasets: Process commodities individually

### Extending the Model

- Add exogenous variables (GDP, fuel prices, etc.) to Prophet/SARIMAX
- Ensemble forecasts: Average Prophet, SARIMAX, and LSTM predictions
- Dynamic thresholds: Flag forecasts when confidence intervals widen significantly

In [ ]:
# Create output folder
OUTPUT_DIR = './outputs/forecasts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Export prophet results
if len(prophet_df) > 0:
    prophet_df.to_csv(f'{OUTPUT_DIR}/prophet_metrics.csv')
    print(f"✓ Saved: {OUTPUT_DIR}/prophet_metrics.csv")

# Export SARIMAX results
if len(sarimax_df) > 0:
    sarimax_df.to_csv(f'{OUTPUT_DIR}/sarimax_metrics.csv')
    print(f"✓ Saved: {OUTPUT_DIR}/sarimax_metrics.csv")

# Export LSTM results
if len(lstm_df) > 0:
    lstm_df.to_csv(f'{OUTPUT_DIR}/lstm_metrics.csv')
    print(f"✓ Saved: {OUTPUT_DIR}/lstm_metrics.csv")

# Create combined summary
summary = pd.DataFrame({
    'Prophet_RMSE': prophet_df['RMSE'].mean() if len(prophet_df) > 0 else None,
    'Prophet_MAPE': prophet_df['MAPE'].mean() if len(prophet_df) > 0 else None,
    'SARIMAX_RMSE': sarimax_df['RMSE'].mean() if len(sarimax_df) > 0 else None,
    'SARIMAX_MAPE': sarimax_df['MAPE'].mean() if len(sarimax_df) > 0 else None,
    'LSTM_RMSE': lstm_df['RMSE'].mean() if len(lstm_df) > 0 else None,
    'LSTM_MAPE': lstm_df['MAPE'].mean() if len(lstm_df) > 0 else None,
}, index=[0])

summary.to_csv(f'{OUTPUT_DIR}/summary_metrics.csv', index=False)
print(f"✓ Saved: {OUTPUT_DIR}/summary_metrics.csv")

print(f"\nAll results exported to: {OUTPUT_DIR}")

## 11. Export Results

Save forecasts and metrics for reporting and further analysis.

In [ ]:
# Create summary tables for each model
print("\n" + "="*70)
print("PROPHET PERFORMANCE")
print("="*70)
prophet_df = pd.DataFrame(all_results['Prophet']).T
if len(prophet_df) > 0:
    print(prophet_df.round(4))
    print(f"\nAverage RMSE: {prophet_df['RMSE'].mean():.2f}")
    print(f"Average MAPE: {prophet_df['MAPE'].mean():.4f}")

print("\n" + "="*70)
print("SARIMAX PERFORMANCE")
print("="*70)
sarimax_df = pd.DataFrame(all_results['SARIMAX']).T
if len(sarimax_df) > 0:
    print(sarimax_df.round(4))
    print(f"\nAverage RMSE: {sarimax_df['RMSE'].mean():.2f}")
    print(f"Average MAPE: {sarimax_df['MAPE'].mean():.4f}")

print("\n" + "="*70)
print("LSTM PERFORMANCE")
print("="*70)
lstm_df = pd.DataFrame(all_results['LSTM']).T
if len(lstm_df) > 0:
    print(lstm_df.round(4))
    print(f"\nAverage RMSE: {lstm_df['RMSE'].mean():.2f}")
    print(f"Average MAPE: {lstm_df['MAPE'].mean():.4f}")

# Comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison
rmse_data = {
    'Prophet': prophet_df['RMSE'].mean() if len(prophet_df) > 0 else None,
    'SARIMAX': sarimax_df['RMSE'].mean() if len(sarimax_df) > 0 else None,
    'LSTM': lstm_df['RMSE'].mean() if len(lstm_df) > 0 else None,
}
rmse_data = {k: v for k, v in rmse_data.items() if v is not None}
axes[0].bar(rmse_data.keys(), rmse_data.values(), color=['green', 'orange', 'red'])
axes[0].set_title('Average RMSE by Model', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE')
axes[0].grid(True, alpha=0.3, axis='y')

# MAPE comparison
mape_data = {
    'Prophet': prophet_df['MAPE'].mean() if len(prophet_df) > 0 else None,
    'SARIMAX': sarimax_df['MAPE'].mean() if len(sarimax_df) > 0 else None,
    'LSTM': lstm_df['MAPE'].mean() if len(lstm_df) > 0 else None,
}
mape_data = {k: v for k, v in mape_data.items() if v is not None}
axes[1].bar(mape_data.keys(), mape_data.values(), color=['green', 'orange', 'red'])
axes[1].set_title('Average MAPE by Model', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAPE (%)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Results Summary

Aggregate and compare performance across all models and commodities.

In [ ]:
# Initialize results storage
all_results = {
    'Prophet': {},
    'SARIMAX': {},
    'LSTM': {}
}

# Run all models on all commodities
print("="*70)
print("RUNNING BATCH FORECASTING FOR ALL COMMODITIES")
print("="*70)

for commodity in sorted(commodities_data.keys()):
    print(f"\n[{commodity.upper()}]")
    
    train_data = train_test_splits[commodity]['train']
    test_data = train_test_splits[commodity]['test']
    
    # Skip if insufficient data
    if len(test_data) == 0:
        print("  ⚠️  No test data available")
        continue
    
    # === PROPHET ===
    try:
        prophet_model, prophet_forecast = forecast_with_prophet(train_data)
        if prophet_model is not None:
            prophet_test_forecast = prophet_forecast[-len(test_data):]['yhat'].values
            prophet_metrics = calculate_metrics(test_data['ttokm'].values, prophet_test_forecast)
            all_results['Prophet'][commodity] = prophet_metrics
            print(f"  ✓ Prophet   - RMSE: {prophet_metrics['RMSE']:.2f} | MAPE: {prophet_metrics['MAPE']:.4f}")
    except Exception as e:
        print(f"  ✗ Prophet failed: {str(e)[:50]}")
    
    # === SARIMAX ===
    try:
        sarimax_model, sarimax_forecast, _, _ = forecast_with_sarimax(train_data)
        if sarimax_model is not None:
            sarimax_test_forecast = sarimax_forecast['mean'].values[:len(test_data)]
            sarimax_metrics = calculate_metrics(test_data['ttokm'].values, sarimax_test_forecast)
            all_results['SARIMAX'][commodity] = sarimax_metrics
            print(f"  ✓ SARIMAX  - RMSE: {sarimax_metrics['RMSE']:.2f} | MAPE: {sarimax_metrics['MAPE']:.4f}")
    except Exception as e:
        print(f"  ✗ SARIMAX failed: {str(e)[:50]}")
    
    # === LSTM ===
    try:
        lstm_model, lstm_forecast = forecast_with_lstm(train_data)
        if lstm_model is not None:
            evaluate_length = min(len(lstm_forecast), len(test_data))
            lstm_test_actual = test_data['ttokm'].values[:evaluate_length]
            lstm_test_forecast = lstm_forecast[:evaluate_length]
            lstm_metrics = calculate_metrics(lstm_test_actual, lstm_test_forecast)
            all_results['LSTM'][commodity] = lstm_metrics
            print(f"  ✓ LSTM     - RMSE: {lstm_metrics['RMSE']:.2f} | MAPE: {lstm_metrics['MAPE']:.4f}")
    except Exception as e:
        print(f"  ✗ LSTM failed: {str(e)[:50]}")

print("\n" + "="*70)
print("BATCH FORECASTING COMPLETE")

## 9. Batch Forecasting for All Commodities

Run all models across all 14 commodity groups and aggregate results.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Plot training data
train_data = train_test_splits[commodity]['train']
ax.plot(train_data['date'], train_data['ttokm'], label='Training Data', color='steelblue', linewidth=2)

# Plot test data (actual)
test_data = train_test_splits[commodity]['test']
if len(test_data) > 0:
    ax.plot(test_data['date'], test_data['ttokm'], label='Test Data (Actual)', 
            color='black', linewidth=2.5, marker='o', markersize=4)
    
    # Plot Prophet forecast (full test length)
    if prophet_model is not None:
        prophet_test_forecast = prophet_forecast[-len(test_data):]['yhat'].values
        ax.plot(test_data['date'], prophet_test_forecast, label='Prophet Forecast', 
                color='green', linewidth=1.5, linestyle='--', alpha=0.8)
    
    # Plot SARIMAX forecast (may be shorter)
    if sarimax_model is not None:
        sarimax_length = min(len(sarimax_forecast), len(test_data))
        sarimax_test_forecast = sarimax_forecast['mean'].values[:sarimax_length]
        ax.plot(test_data['date'][:sarimax_length], sarimax_test_forecast, 
                label=f'SARIMAX Forecast ({sarimax_length} weeks)', 
                color='orange', linewidth=1.5, linestyle='--', alpha=0.8)
    
    # Plot LSTM forecast (may be shorter)
    if lstm_model is not None:
        lstm_length = min(len(lstm_forecast), len(test_data))
        ax.plot(test_data['date'][:lstm_length], lstm_forecast[:lstm_length], 
                label=f'LSTM Forecast ({lstm_length} weeks)', 
                color='red', linewidth=1.5, linestyle='--', alpha=0.8)

ax.set_title(f'{commodity.upper()} - Model Forecast Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('TTOKM', fontsize=12)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Forecast Visualization

Compare model outputs visually on test set.

In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calculate forecast accuracy metrics"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape
    }

def evaluate_prophet_forecast(test_df, prophet_forecast):
    """Extract test period forecast from Prophet and evaluate"""
    try:
        # Get forecast for test period
        forecast_values = prophet_forecast[-len(test_df):]['yhat'].values
        actual_values = test_df['ttokm'].values
        return calculate_metrics(actual_values, forecast_values)
    except:
        return None

# Evaluate Prophet on test set
if prophet_model is not None and len(train_test_splits[commodity]['test']) > 0:
    test_data = train_test_splits[commodity]['test']
    prophet_metrics = evaluate_prophet_forecast(test_data, prophet_forecast)
    print("PROPHET Test Metrics:")
    for metric, value in prophet_metrics.items():
        print(f"  {metric}: {value:.4f}")

# Evaluate SARIMAX on test set
if sarimax_model is not None and len(train_test_splits[commodity]['test']) > 0:
    test_data = train_test_splits[commodity]['test']
    # SARIMAX forecast may be shorter than test set - evaluate on common length
    evaluate_length = min(len(sarimax_forecast), len(test_data))
    forecast_values = sarimax_forecast['mean'].values[:evaluate_length]
    actual_values = test_data['ttokm'].values[:evaluate_length]
    sarimax_metrics = calculate_metrics(actual_values, forecast_values)
    print("\nSARIMAX Test Metrics:")
    for metric, value in sarimax_metrics.items():
        print(f"  {metric}: {value:.4f}")

# Evaluate LSTM on test set
if lstm_model is not None and len(train_test_splits[commodity]['test']) > 0:
    test_data = train_test_splits[commodity]['test']
    # LSTM generates forecast for forecast_weeks ahead (may not match test length exactly)
    evaluate_length = min(len(lstm_forecast), len(test_data))
    actual_values = test_data['ttokm'].values[:evaluate_length]
    forecast_values = lstm_forecast[:evaluate_length]
    lstm_metrics = calculate_metrics(actual_values, forecast_values)
    print("\nLSTM Test Metrics:")
    for metric, value in lstm_metrics.items():
        print(f"  {metric}: {value:.4f}")

## 7. Model Evaluation & Comparison

Evaluate all models on test set and compare metrics.

In [ ]:
def create_lstm_sequences(data, lookback=52):
    """
    Create sequences for LSTM training (lookback = 52 weeks ~ 1 year)
    """
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback])
    return np.array(X), np.array(y)

def forecast_with_lstm(train_df, forecast_weeks=FORECAST_WEEKS, lookback=52):
    """
    Build and train LSTM model for forecasting
    """
    try:
        # Prepare data
        data = train_df['ttokm'].values.reshape(-1, 1)
        scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_data = scaler.fit_transform(data)
        
        # Create sequences
        X, y = create_lstm_sequences(scaled_data.flatten(), lookback=lookback)
        
        if len(X) < 10:
            print("Not enough data for LSTM training")
            return None, None
        
        # Build LSTM model
        model = Sequential([
            LSTM(50, activation='relu', input_shape=(lookback, 1), return_sequences=True),
            Dropout(0.2),
            LSTM(50, activation='relu'),
            Dropout(0.2),
            Dense(25, activation='relu'),
            Dense(1)
        ])
        
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
        
        # Train model (verbose=0 for quiet training)
        model.fit(X.reshape(X.shape[0], X.shape[1], 1), y, epochs=50, batch_size=16, verbose=0)
        
        # Generate forecast
        last_sequence = scaled_data[-lookback:].reshape(1, lookback, 1)
        forecasts = []
        
        for _ in range(forecast_weeks):
            pred = model.predict(last_sequence, verbose=0)
            forecasts.append(pred[0, 0])
            # Shift window
            last_sequence = np.append(last_sequence[:, 1:, :], [[[pred[0, 0]]]], axis=1)
        
        # Inverse transform
        forecast_values = scaler.inverse_transform(np.array(forecasts).reshape(-1, 1)).flatten()
        
        return model, forecast_values
    except Exception as e:
        print(f"LSTM error: {e}")
        return None, None

# Test LSTM on first commodity
print("Training LSTM on first commodity (this may take a moment)...")
lstm_model, lstm_forecast = forecast_with_lstm(train_data)

if lstm_model is not None:
    print(f"✓ LSTM model trained for {commodity}")
    print(f"Forecast values: {lstm_forecast[:5]} (first 5 weeks)")
else:
    print("LSTM training skipped")

## 6. LSTM Model

Deep learning approach for capturing non-linear patterns.

In [ ]:
def forecast_with_sarimax(train_df, forecast_weeks=FORECAST_WEEKS):
    """
    Use auto_arima to find optimal SARIMAX parameters
    """
    try:
        # Prepare time series data
        y = train_df['ttokm'].values
        
        # Auto find ARIMA parameters (suppress verbose output)
        auto_model = pm.auto_arima(
            y,
            start_p=0, max_p=5,
            start_q=0, max_q=5,
            start_P=0, max_P=2,
            start_Q=0, max_Q=2,
            m=52,  # Weekly seasonality (52 weeks/year)
            seasonal=True,
            stepwise=True,
            trace=False,
            error_action='ignore',
            suppress_warnings=True
        )
        
        # Fit SARIMAX model
        order = auto_model.order
        seasonal_order = auto_model.seasonal_order
        model = SARIMAX(y, order=order, seasonal_order=seasonal_order)
        fitted_model = model.fit(disp=False)
        
        # Generate forecast
        forecast = fitted_model.get_forecast(steps=forecast_weeks)
        forecast_df = forecast.summary_frame()
        
        return fitted_model, forecast_df, order, seasonal_order
    except Exception as e:
        print(f"SARIMAX error: {e}")
        return None, None, None, None

# Test SARIMAX on first commodity
print("Testing SARIMAX on first commodity...")
sarimax_model, sarimax_forecast, order, seasonal_order = forecast_with_sarimax(train_data)

if sarimax_model is not None:
    print(f"✓ SARIMAX model trained for {commodity}")
    print(f"Order (p,d,q): {order}")
    print(f"Seasonal Order (P,D,Q,s): {seasonal_order}")
    print(f"Forecast shape: {sarimax_forecast.shape}")

## 5. SARIMAX Model

AutoML approach to find optimal ARIMA parameters using pmdarima.

In [ ]:
def forecast_with_prophet(train_df, forecast_weeks=FORECAST_WEEKS):
    """
    Fit Prophet model and generate forecast
    """
    try:
        # Prepare data for Prophet (requires 'ds' and 'y' columns)
        prophet_df = train_df[['date', 'ttokm']].copy()
        prophet_df.columns = ['ds', 'y']
        
        # Fit model with weekly seasonality
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            changepoint_prior_scale=0.05,
            seasonality_prior_scale=10,
            interval_width=0.95
        )
        
        with suppress_output():
            model.fit(prophet_df)
        
        # Create future dataframe (weekly frequency)
        future = model.make_future_dataframe(periods=forecast_weeks, freq='W')
        forecast = model.predict(future)
        
        return model, forecast
    except Exception as e:
        print(f"Prophet error: {e}")
        return None, None

from contextlib import contextmanager
import sys
from io import StringIO

@contextmanager
def suppress_output():
    save_stdout = sys.stdout
    save_stderr = sys.stderr
    sys.stdout = StringIO()
    sys.stderr = StringIO()
    try:
        yield
    finally:
        sys.stdout = save_stdout
        sys.stderr = save_stderr

# Test Prophet on one commodity
print("Testing Prophet on first commodity...")
commodity = list(commodities_data.keys())[0]
train_data = train_test_splits[commodity]['train']
prophet_model, prophet_forecast = forecast_with_prophet(train_data)

if prophet_model is not None:
    print(f"✓ Prophet model trained for {commodity}")
    print(f"Forecast shape: {prophet_forecast.shape}")
    print(f"Forecast columns: {prophet_forecast.columns.tolist()}")

## 4. Prophet Model

Facebook's Prophet for time series forecasting with trend and seasonality.

In [ ]:
# ===== FORECASTING PARAMETERS =====
FORECAST_WEEKS = 26  # 6 months ~ 26 weeks (adjust for 3-18 month range)
TEST_SIZE = 52       # Last 1 year for testing

# Split data into train/test
def prepare_train_test_split(df, test_size=TEST_SIZE):
    """
    Split time series data into train and test sets.
    test_size: number of weeks to reserve for testing
    """
    train = df[:-test_size].copy() if len(df) > test_size else df.copy()
    test = df[-test_size:].copy() if len(df) > test_size else pd.DataFrame()
    return train, test

# Prepare splits for all commodities
train_test_splits = {}
for commodity, df in commodities_data.items():
    train, test = prepare_train_test_split(df, test_size=TEST_SIZE)
    train_test_splits[commodity] = {'train': train, 'test': test}
    print(f"{commodity}: Train={len(train)} weeks | Test={len(test)} weeks")

## 3. Forecasting Configuration

Define train/test split and forecast horizon.

In [ ]:
# Summary statistics for each commodity
summary_stats = {}
for commodity, df in commodities_data.items():
    summary_stats[commodity] = {
        'rows': len(df),
        'date_range': f"{df['date'].min().date()} to {df['date'].max().date()}",
        'missing_values': df['ttokm'].isna().sum(),
        'mean_ttokm': df['ttokm'].mean(),
        'std_ttokm': df['ttokm'].std(),
        'min_ttokm': df['ttokm'].min(),
        'max_ttokm': df['ttokm'].max(),
        'years_covered': df['year'].max() - df['year'].min() + 1
    }

summary_df = pd.DataFrame(summary_stats).T
print(summary_df.round(2))

# Visualize time series for all commodities
fig, axes = plt.subplots(len(commodities_data), 1, figsize=(14, 3*len(commodities_data)))
if len(commodities_data) == 1:
    axes = [axes]

for idx, (commodity, df) in enumerate(commodities_data.items()):
    axes[idx].plot(df['date'], df['ttokm'], linewidth=1.5, color='steelblue')
    axes[idx].set_title(f'{commodity.upper()} - Weekly TTOKM', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('TTOKM', fontsize=10)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Exploratory Data Analysis

Visualize data quality and trends across commodities.

In [ ]:
def convert_week_to_datetime(year, week):
    """Convert ISO year and week to datetime (Monday of that week)"""
    try:
        # ISO week format: year-Www (e.g., 2023-W01)
        date_str = f"{year}-W{int(week):02d}-1"
        return pd.to_datetime(date_str, format='%G-W%V-%u')
    except:
        return pd.NaT

def load_and_prepare_commodity_data(commodity_name):
    """
    Load commodity CSV file and prepare time series data.
    
    Expected file format:
    - VJA: year
    - VMO: month
    - VKW: calendar week
    - NHM: product classification (aggregated)
    - TTOKM: thousand ton-kilometers
    """
    # Try different file naming patterns
    possible_names = [
        f'{commodity_name.lower()}2005_2024.csv',
        f'{commodity_name.lower()}.csv',
        f'{commodity_name}.csv'
    ]
    
    filepath = None
    for name in possible_names:
        full_path = os.path.join(DATA_FOLDER, name)
        if os.path.exists(full_path):
            filepath = full_path
            break
    
    if filepath is None:
        print(f"⚠️  WARNING: No file found for {commodity_name}")
        return None
    
    try:
        # Load CSV with expected column names
        df = pd.read_csv(filepath, sep=',')
        
        # Rename columns to standardized names
        df.columns = df.columns.str.upper().str.strip()
        
        # Map Austrian column names to our standard format
        if 'VJA' in df.columns:
            df['year'] = df['VJA']
        if 'VMO' in df.columns:
            df['month'] = df['VMO']
        if 'VKW' in df.columns:
            df['week'] = df['VKW']
        if 'TTOKM' in df.columns:
            df['ttokm'] = df['TTOKM']
        
        # Aggregate by (year, month, week) - sum across product classifications
        df_agg = df.groupby(['year', 'month', 'week']).agg({
            'ttokm': 'sum'
        }).reset_index()
        
        # Create datetime from year and week columns
        df_agg['date'] = df_agg.apply(
            lambda row: convert_week_to_datetime(int(row['year']), int(row['week'])),
            axis=1
        )
        
        # Sort by date
        df_agg = df_agg.sort_values('date').reset_index(drop=True)
        
        # Select relevant columns
        df_agg = df_agg[['date', 'year', 'month', 'week', 'ttokm']].copy()
        
        # Handle missing values
        df_agg['ttokm'] = pd.to_numeric(df_agg['ttokm'], errors='coerce')
        
        print(f"✓ {commodity_name}: {len(df_agg)} weeks | {df_agg['date'].min().date()} to {df_agg['date'].max().date()}")
        return df_agg
    
    except Exception as e:
        print(f"✗ Error loading {commodity_name}: {str(e)}")
        return None

# Load all commodities
commodities_data = {}
for commodity in COMMODITIES:
    data = load_and_prepare_commodity_data(commodity)
    if data is not None:
        commodities_data[commodity] = data

print(f"\n{'='*60}")
print(f"Successfully loaded: {len(commodities_data)}/{len(COMMODITIES)} commodities")

In [ ]:
# Define commodity groups based on available files
COMMODITIES = [
    'wood', 'paper', 'chemicals', 'buildingmaterials', 'energy',
    'scrap', 'metals', 'minerals', 'food_products',
    'textiles', 'machinery', 'vehicles', 'coal', 'fertilizers'
]

# Data folder - adjust path to your data location
DATA_FOLDER = '/Users/Hans/Documents/EMBA/MasterThesis/Data'

# Create data folder if it doesn't exist
os.makedirs(DATA_FOLDER, exist_ok=True)

print(f"Data folder: {DATA_FOLDER}")
print(f"Folder exists: {os.path.exists(DATA_FOLDER)}")

# List available files
available_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.csv')]
print(f"\nAvailable CSV files ({len(available_files)}):")
for f in sorted(available_files):
    print(f"  - {f}")

## 1. Data Loading & Preprocessing

Load all commodity CSV files and convert weekly data to datetime format.

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import pyarrow.parquet as pq

# Time Series Libraries
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
import pmdarima as pm
from prophet import Prophet

# Machine Learning
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Multi-Commodity Time Series Forecasting

**Objective:** Build predictive models for 14 commodity groups using 15 years of weekly transportation data (tonkm).

**Scope:** 
- Data: Year, Month, Calendar Week, TTOKM (thousand ton-kilometers)
- History: ~15 years (weekly intervals)
- Forecast: 3-18 months ahead
- Models: Prophet, SARIMAX, LSTM
- Comparison: Cross-commodity performance metrics

In [ ]:
%pip install -q numpy pandas matplotlib seaborn statsmodels pmdarima prophet tensorflow scikit-learn pyarrow